# Разбор A. Конформеры и геометрия

Один SMILES задаёт связность, но не положение каждого атома. Истрадефиллин может образовать разные пространственные конфигурации вращаемых связей; изображение на экране не выбирает состояние в растворе. Гипотеза: повторное внедрение с разными seed даёт набор локальных геометрий. Для B гибкая боковая цепь делает один конформер особенно слабым основанием для решения.

Добавьте явные H, проверьте наличие параметров MMFF. Сгенерируйте восемь конформеров истрадефиллина с фиксированным seed и оптимизируйте каждый. Запишите код сходимости и энергию, экспортируйте SDF с метками. Проанализируйте только сошедшиеся значения; не скрывайте отказы.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M06-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem
m=Chem.AddHs(Chem.MolFromSmiles(compound('istradefylline')['smiles']))
assert AllChem.MMFFHasAllMoleculeParams(m)
p=AllChem.ETKDGv3();p.randomSeed=42;p.numThreads=1
ids=list(AllChem.EmbedMultipleConfs(m,numConfs=8,params=p));assert ids
energies=AllChem.MMFFOptimizeMoleculeConfs(m,numThreads=1,maxIters=1000)
rows=[{'id':int(i),'status':int(s),'energy_kcal_mol':float(e)} for i,(s,e) in zip(ids,energies)]
display(rows)


[{'id': 0, 'status': 0, 'energy_kcal_mol': -84.77616037591137},
 {'id': 1, 'status': 0, 'energy_kcal_mol': -84.67686013495481},
 {'id': 2, 'status': 0, 'energy_kcal_mol': -80.88543294044968},
 {'id': 3, 'status': 0, 'energy_kcal_mol': -80.92386901821118},
 {'id': 4, 'status': 0, 'energy_kcal_mol': -84.67686002343018},
 {'id': 5, 'status': 0, 'energy_kcal_mol': -80.88543270959772},
 {'id': 6, 'status': 0, 'energy_kcal_mol': -84.77616107986555},
 {'id': 7, 'status': 0, 'energy_kcal_mol': -80.96813653465018}]

Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
with Chem.SDWriter(str(OUT/'conformers.sdf')) as writer:
    for r in rows:
        m.SetProp('MMFF_status',str(r['status']));m.SetProp('energy_kcal_mol',str(r['energy_kcal_mol']));writer.write(m,confId=r['id'])
good=[r for r in rows if r['status']==0]
if not good:raise RuntimeError('No converged conformer')
fig,ax=plt.subplots();ax.bar([str(r['id']) for r in good],[r['energy_kcal_mol']-min(x['energy_kcal_mol'] for x in good) for r in good])
ax.set(xlabel='Conformer',ylabel='Relative MMFF energy / kcal mol^-1');publish_plot(fig,'conformers.png')


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'rows':rows,'converged':len(good),'seed':42,'kind':'calculated','solvent':'none'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M06-L01','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M06-L01\20261118T141556626354Z-A-e8efaf387ad84d0ca50b6374f0f4a8c4.json


Распределение найденных минимумов зависит от начального поиска и не равно частоте в растворе. Сравнение с другими seed проверит полноту поиска, а температурное взвешивание требует ещё одной модели.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?